# LangChain使用之Memory
## 1、Memory概述
### 1.1 为什么需要Memory
大多数的大模型应用程序都会有一个会话接口，允许我们进行多轮的对话，并有一定的上下文记忆能力。

但实际上，模型本身是 不会记忆 任何上下文的，只能依靠用户本身的输入去产生输出。

如何实现记忆功能呢？

实现这个记忆功能，就需要 额外的模块 去保存我们和模型对话的上下文信息，然后在下一次请求时，把所有的历史信息都输入给模型，让模型输出最终结果。

而在 LangChain 中，提供这个功能的模块就称为 Memory(记忆) ，用于存储用户和模型交互的历史信息。

### 1.2 什么是Memory
Memory，是LangChain中用于多轮对话中保存和管理上下文信息（比如文本、图像、音频等）的组件。它让应用能够记住用户之前说了什么，从而实现对话的 上下文感知能力 ，为构建真正智能和上下文感知的链式对话系统提供了基础。

### 1.3 Memory的设计理念
> 1. 输入问题：({"question": ...})
2. 读取历史消息：从Memory中READ历史消息（{"past_messages": [...]}）
3. 构建提示（Prompt)：读取到的历史消息和当前问题会被合并，构建一个新的Prompt
4. 模型处理：构建好的提示会被传递给语言模型进行处理。语言模型根据提示生成一个输出。
5. 解析输出：输出解析器通过正则表达式 regex("Answer: (.*)")来解析，返回一个回答（{"answer":...}）给用户
6. 得到回复并写入Memory：新生成的回答会与当前的问题一起写入Memory，更新对话历史。Memory会存储最新的对话内容，为后续的对话提供上下文支持

### 1.4 不使用Memory模块，如何拥有记忆？
不借助LangChain情况下，我们如何实现大模型的记忆能力？

思考：通过 messages 变量，不断地将历史的对话信息追加到对话列表中，以此让大模型具备上下文记忆能力。

In [1]:
from langchain_ollama import ChatOllama
# 创建大模型实例
llm = ChatOllama(model="qwen:7b")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
def chat_with_model(question):
    # 步骤一：初始化消息
    chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一位人工智能小助手"),
    ("human","{question}")
    ])
    # 步骤二：定义一个循环体：
    while True:
        # 步骤三：调用模型
        chain = chat_prompt_template | llm
        response = chain.invoke({"question": question})
        # 步骤四：获取模型回答
        print(f"模型回答: {response.content}")
        # 询问用户是否还有其他问题
        user_input = input("您还有其他问题想问嘛？(输入'退出'结束对话)")
        # 设置结束循环的条件
        if(user_input == "退出"):
            break
        # 步骤五：记录用户回答
        chat_prompt_template.messages.append(AIMessage(content=response.content))
        chat_prompt_template.messages.append(HumanMessage(content=user_input))

chat_with_model("你好")

模型回答: 你好！有什么我能帮你的？
模型回答: 冰封万里雪皑皑，大地银装素裹鲜。

炉火熠熠驱严寒，红梅傲骨显春意。

踏雪寻梅冬韵长，笔端生花寄情深。
模型回答: 当然可以！"冰封万里雪皑皑" 这句话的白话解释是：

"冰雪覆盖了大半个地球，洁白无瑕。"

这样你就理解了诗句的第一句含义。


## 2、基础Memory模块的使用
### 2.1 Memory模块的设计思路
如何设计Memory模块？

- 层次1(最直接的方式)：保留一个聊天消息列表
- 层次2(简单的新思路)：只返回最近交互的k条消息
- 层次3(稍微复杂一点)：返回过去k条消息的简洁摘要
- 层次4(更复杂)：从存储的消息中提取实体，并且仅返回有关当前运行中引用的实体的信息